# 載入常用套件

In [3]:
import pandas as pd
import numpy as np
import os
import json
import csv
import sqlite3
import time
import random
import re
import requests
import emoji
from abc import abstractmethod
from DrissionPage import ChromiumPage
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime

# CrawlerManager

In [ ]:
class BaseCrawler:
    def __init__(self, topic_name, save_folder):
        self.topic_name = topic_name
        self.save_folder = save_folder
        if not os.path.exists(save_folder):
            os.makedirs(save_folder)

    def save_data(self, data, filename):
        file_path = os.path.join(self.save_folder, f"{self.topic_name}_{filename}.json")
        try:
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=4)
            print(f"成功儲存: {file_path}")
        except Exception as e:
            print(f"存檔失敗: {e}")

    @abstractmethod
    def format_data(self, post, comments):
        pass

## Dcard 爬蟲

In [ ]:
class dcardcrawler(BaseCrawler):
    default_pages = 1

    def __init__(self, save_folder="dcard_data"):
        super().__init__(topic_name="dcard", save_folder=save_folder)

    @classmethod
    def create(cls, save_folder="dcard_data"):
        return cls(save_folder=save_folder)
    
    # 網址建構
    def _build_urls(self, mode, query, forum=None):
        """
        依據模式生成前端過驗證網址與後端 api 網址
        """
        mode = mode.lower()

        if mode == "topic":
            # 話題追蹤
            front_url =f"https://www.dcard.tw/topics/{query}?tab=latest"
            api_url = (
                f"https://www.dcard.tw/service/api/v3/search/posts?"
                f"query={query}&field=topics&highlight=false&sort=latest&country=TW&nsfw=true&platform=web"
            )
        elif mode == "search":
            # 支援限定看板或全站搜尋
            if forum:
                front_url = f"https://www.dcard.tw/f/{forum}?tab=latest" # 先到該看板過驗證
                api_url = (
                    f"https://www.dcard.tw/service/api/v3/search/posts?"
                    f"query={query}&forum={forum}&field=all&highlight=false&sort=latest&country=TW&nsfw=true&platform=web"
                )
            else:
                front_url = f"https://www.dcard.tw/search/posts?query={query}&sort=latest"
                api_url = (
                    f"https://www.dcard.tw/service/api/v3/search/posts?"
                    f"query={query}&field=all&highlight=false&sort=latest&country=TW&nsfw=true&platform=web"
                )

        elif mode == "forum":
            target_forum = forum if forum else query
            front_url = f"https://www.dcard.tw/f/{target_forum}?tab=latest"
            api_url = f"https://www.dcard.tw/service/api/v3/forums/{target_forum}/posts?sort=new"
                
        else:
            raise ValueError(f"未知的 Dcard 爬取模式: {mode}")

        return front_url, api_url
    
    def page_to_json(self, page):
        match = re.search(r'<pre>(.*?)</pre>', page.html, re.S)
        if not match:
            return None
        try:
            return json.loads(match.group(1))
        except:
            return None
        
    def extract_posts(self, data, mode):
        # 如果是純看板模式(forum)
        if not data:
            return []
        
        if isinstance(data, list):
            return data

        posts = []
        if isinstance(data, dict):
            if "widgets" in data:
                widgets = data.get("widgets", [])
                for widget in widgets:
                    if not isinstance(widget, dict):
                        continue
                    items = widget.get("forumlist", {}).get("items", [])
                    for item in items:
                        if isinstance(item, dict) and "post" in item:
                            posts.append(item.get("post"))
                return posts
            
        items = data.get("items", [])
        for item in items:
            if "searchPost" in item:
                posts.append(item.get("searchPost", {}).get("post", {}))
            elif "post" in item:
                posts.append(item.get("post", {}))
            elif isinstance(item, dict) and "id" in item:
                posts.append(item)
        return posts
            
    def fetch_comments(self, page, post_id, comment_limit):
        comment_url = f"https://www.dcard.tw/service/api/v3/posts/{post_id}/comments?limit={comment_limit}&sort=oldest"
        page.get(comment_url)
        time.sleep(random.uniform(1, 2))

        data = self.page_to_json(page)
        if not data:
            return []
        if isinstance(data, dict):
            return data.get("items", data.get("comments", []))
        elif isinstance(data, list):
            return data
        return []
    
    def format_data(self, post, comments):
        # 時間
        raw_time = post.get('createdAt', '')
        try:
            dt = datetime.fromisoformat(raw_time.replace('Z', '+00:00'))
            post_time = dt.strftime("%Y-%m-%d")
        except:
            post_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S") # 容錯處理

        # 作者
        author = post.get('school') or post.get('department')
        if post.get('anonymous', True) or not author:
            author = "Anonymous"

        # 內文
        content = post.get("meta", {}).get("annotation")
        if not content:
            content = post.get("excerpt", "")

        # 留言
        formatted_comments = []
        for c in comments:
            if not isinstance(c, dict): continue
            # 抓取留言學校/卡稱，若匿名則給 Anonymous
            c_author = c.get("school") or c.get("department")
            if c.get("anonymous", True) or not c_author:
                c_author = "Anonymous"
                
            formatted_comments.append({
                "comment_author": c_author,
                "comment": c.get("content", "")
            })

        return {
            "platform": "Dcard",
            "post_time": post_time,
            "author": author,
            "title": post.get("title", "無標題"),
            "content": content,
            "total_reac": post.get('likeCount', 0),
            "comment_count": post.get('commentCount', 0),
            "comments_data": formatted_comments
        }
        
        
    def crawl_all_pages(self, page, base_url, max_pages, comment_limit, mode):
        next_key = None
        all_data = []

        for page_num in range(max_pages):
            print(f"\n 第 {page_num+1} 頁 ")

            if next_key:
                url = f"{base_url}&nextKey={next_key}" if "nextKey" not in base_url else base_url
            else:
                url = base_url

            page.get(url)
            time.sleep(random.uniform(2, 4))

            data = self.page_to_json(page)
            if not data:
                print("解析失敗")
                break
            posts = self.extract_posts(data, mode)
            print(f"抓到 {len(posts)} 篇文章")

            for i, post in enumerate(posts, 1):
                post_id = post.get("id")
                if not post_id:
                    continue
                
                # 抓原始留言
                comments = self.fetch_comments(page, post_id, comment_limit)
                print(f"留言 ({len(comments)} 則):")

                for c in comments:
                    if not isinstance(c, dict):
                        continue
                    floor = c.get("floor", "?")
                    text = c.get("content", "")
                    print(f"  {floor}F: {text[:20]}...")
                    time.sleep(random.uniform(2, 4))
                    # comment_list.append({
                    #     "comment": text
                    # })

                formatted_post = self.format_data(post, comments)
                all_data.append(formatted_post)

                time.sleep(random.uniform(1, 2))

                    # if post_time < "2025-01-01": 
                    #     print("已到達目標日期，停止抓取")
                    #     return all_data

                # 取得下一頁密鑰
            if isinstance(data, dict):
                next_key = data.get("nextKey")
            elif isinstance(data, list) and data:
                # 純看板模式的分頁通常是用最後一篇文章的 id 當作 before 參數
                next_key = data[-1].get("id")
                base_url = base_url.split('&before=')[0] + f"&before={next_key}"
            else:
                next_key = None

            if not next_key:
                print("已經到達最後一頁")
                break
                
        return all_data

    def run(self, mode="topic", query=None, forum=None, pages=2, comment_limit=2, **kwargs):
        """
        泛用型
        param mode: 'topic' (話題) 或 'search' (搜尋) 或 'forum' (純看板)
        param query: 關鍵字或話題名稱 (例如 'PokemonGO')
        param forum: 限定看板名稱 (例如 'pokemon')
        """
        if not query and mode != "forum":
            raise ValueError("在當前模式下，必須提供 query 參數")
        
        # 透過建構器取得對應網址
        front_url, base_url = self._build_urls(mode, query, forum)

        print(f"開始執行 dcard 爬蟲 [模式: {mode}]")
        print(f"前端網址: {front_url}")

        page = ChromiumPage()
        page.get(front_url)

        input("過驗證出現正常畫面後回 VS code 按下 Enter")

        # 開始爬取
        data = self.crawl_all_pages(page, base_url, pages, comment_limit, mode)

        file_name = f"{mode}_{query or forum}"
        self.save_data(data, f"{file_name}_result")

        page.quit()

## 巴哈 哈拉版爬蟲

In [ ]:
from urllib.parse import parse_qs, urlparse


class bahacrawler(BaseCrawler):
    
    default_pages = 1
    BASE_URL = "https://forum.gamer.com.tw"

    def __init__(self, board, save_folder="baha_data"):
        super().__init__(topic_name=board, save_folder=save_folder)
        self.board = board
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36"
    }
    @classmethod
    def create(cls, board, save_folder="baha_data"):
        return cls(board=board, save_folder=save_folder)
    
    # 建立看板型url
    def build_board_url(self, board_id, page=1):
        return f"{self.BASE_URL}/B.php?page={page}&bsn={board_id}"

    # 建立文章型url
    def build_article_url(self, board_id, article_id, page=1):
        return f"{self.BASE_URL}/C.php?page={page}&bsn={board_id}&snA={article_id}"
    
    def crawl_board(self, board_id, max_pages):
        return self.parse_article(board_id, article_id=max_pages, page=1)

    def crawl_article(self, board_id, article_id):
        return self.parse_article(board_id, article_id, page=1)
    
    def fetch_article_ids(self, board_id, page=1):
        url = self.build_board_url(board_id, page)
        try:
            res = requests.get(url, headers=self.headers, timeout=10)
            if res.status_code != 200:
                print(f" 看板列表請求失敗，狀態碼: {res.status_code}")
                return []
            
            soup = BeautifulSoup(res.text, "html.parser")
            # 放寬選擇器，直接抓取文章標題連結
            links = soup.select(".b-list__main__title")
            
            article_ids = []
            for link in links:
                href = link.get('href', '')
                if 'snA=' in href:
                    # 使用標準解析器提取 snA，避免 split 失敗
                    parsed_url = urlparse(href)
                    query_params = parse_qs(parsed_url.query)
                    aid = query_params.get('snA', [None])[0]
                    if aid and aid not in article_ids:
                        article_ids.append(aid)
            
            print(f" 第 {page} 頁成功解析出 {len(article_ids)} 篇文章 ID")
            return article_ids
        except Exception as e:
            print(f" fetch_article_ids 發生錯誤: {e}")
            return []
    
    def get_total_pages(self, board_id, article_id):
        url = self.build_article_url(board_id, article_id, page=1)
        try:
            res = requests.get(url, headers=self.headers)
            soup = BeautifulSoup(res.text, "html.parser")
            
            # 尋找分頁元件
            pagination = soup.select_one(".BH-pagebtnA")
            if not pagination:
                return 1
            max_page = 1
            for a_tag in pagination.select("a"):
                href = a_tag.get("href", "")
                parsed_url = urlparse(href)
                query_params = parse_qs(parsed_url.query)
                page_val = query_params.get('page', [None])[0]
                if page_val and page_val.isdigit():
                    max_page = max(max_page, int(page_val))
                
            return max_page
        except Exception as e:
            print(f" 抓取文章總頁數失敗 ({e})，預設為 1 頁")
            return 1

    def format_data(self, post, main_title):
        # 標題
        try:
            title_ele = post.select_one("h1.c-post__header__title")
            title = title_ele.text.strip() if title_ele else main_title
        except:
            title = main_title

        # 作者
        try:
            author_ele = post.select_one("a.username")
            author = author_ele.text.strip() if author_ele else "Anonymous"
        except:
            author = "Anonymous"

        # 時間
        try:
            time_ele = post.select_one('.c-post__header__info a.edittime')

            if time_ele and time_ele.has_attr("data-mtime"):
                raw_time = time_ele["data-mtime"]
                
                dt = datetime.strptime(raw_time, "%Y-%m-%d %H:%M:%S")
                post_time = dt.strftime("%Y-%m-%d")
            else:
                post_time = "未成功抓取時間"
        except Exception as e:
            post_time = f"解析失敗: {e}"

        # 互動數(任何)
        try:
            like_tag = post.select_one("a.count.tippy-gpbp-list")
            total_reac = int(like_tag.text.strip()) if like_tag else 0
        except:
            total_reac = 0

        # 內文
        try:
            content = post.select_one("div.c-article__content").text.strip()
        except:
            content = ""
        
        # 留言
        formatted_comments = []
        for c in post.select(".c-reply__item"):
            try:
                # 巴哈留言作者
                c_author_ele = c.select_one(".reply-content__user")
                c_author = c_author_ele.text.strip().replace("：", "") if c_author_ele else "Anonymous"
                
                c_text = c.select_one(".reply-content__article").text.strip()
                
                formatted_comments.append({
                    "comment_author": c_author,
                    "comment": c_text
                })
            except:
                continue

        return {
            "platform": "Bahamut",
            "post_time": post_time,
            "author": author,
            "title": title,
            "content": content,
            "total_reac": total_reac,
            "comment_count": len(formatted_comments),
            "comments_data": formatted_comments
        }

    def parse_article(self, board_id, article_id, page=1):
        url = self.build_article_url(board_id, article_id, page)

        try:
            res = requests.get(url, headers=self.headers)
            soup = BeautifulSoup(res.text, "html.parser")
        except Exception as e:
            print(f"網路請求失敗: {e}")
            return []
        
        all_floor_data = []
        
        posts = soup.select(".c-section")

        main_title_ele = soup.select_one("h1.c-post__header__title")
        main_title = main_title_ele.text.strip() if main_title_ele else "巴哈討論串"
        for post in posts:
            # 呼叫 format_data
            floor_data = self.format_data(post, main_title)
            all_floor_data.append(floor_data)
            
        return all_floor_data

    def run(self, board_id, start_board_page=1, end_board_page=1, reply_pages=2):
        """
        執行動態分頁爬取
        :param board_id: 看板代號 (bsn)
        :param start_board_page: 開始爬取的看板頁碼
        :param end_board_page: 結束爬取的看板頁碼
        :param reply_pages: 預計爬取的文章內回覆頁數
        """
        # 參數檢查
        if start_board_page < 1 or end_board_page < start_board_page:
            print("錯誤：頁碼設定不合法。請確保 start_board_page >= 1 且 end_board_page >= start_board_page")
            return []
        
        all_data = []

        for p in range(start_board_page, end_board_page + 1):
            print(f"正在掃描看板第 {p} 頁列表...")
            aids = self.fetch_article_ids(board_id, p)

            for aid in aids:
                print(f"  -> 處理文章 ID: {aid}")

                # 判斷回覆頁數
                total_pages = self.get_total_pages(board_id, aid)
                print(f"     共 {total_pages} 頁回覆")

                if total_pages > 3:
                    start_page = total_pages
                    end_page = max(total_pages - reply_pages + 1, 1)
                    target_pages = list(range(start_page, end_page - 1, -1  ))
                else:
                    target_pages = list(range(1, min(total_pages, reply_pages) + 1))

                for page in target_pages:
                    print(f"     -> 抓取第 {page} 頁回覆...")
                    page_data = self.parse_article(board_id, aid, page=page)
                    all_data.extend(page_data)
                    time.sleep(random.uniform(2, 4))

        self.save_data(all_data, f"baha_{board_id}")
        return all_data

## ptt 爬蟲

In [ ]:
class pttcrawler(BaseCrawler):

    BASE_URL = "https://www.ptt.cc"
    default_pages = 1

    def __init__(self, board, save_folder="ptt_data"):
        super().__init__(topic_name=board, save_folder=save_folder)
        self.board = board

    @classmethod
    def create(cls, board, save_folder="ptt_data"):
        return cls(board=board, save_folder=save_folder)

    def build_board_url(self, board, index=None):
            if index:
                return f"{self.BASE_URL}/bbs/{board}/index{index}.html"
            else:
                return f"{self.BASE_URL}/bbs/{board}/index.html"

    def build_article_url(self, href):
        return f"{self.BASE_URL}{href}"

    def get_response(self, url, retries=3):
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)",
            "Referer": self.BASE_URL
        }
        cookies = {"over18": "1"}

        for i in range(retries):
            try:
                res = requests.get(url, headers=headers, cookies=cookies, timeout=10)
                if res.status_code == 200:
                    return res
            except Exception as e:
                print(f"Retry {i+1}: {e}")
                time.sleep(2)
        return None
    
    def format_data(self, raw_time, author, title, content, like, unlike, push_list):
        # 時間
        try:
            dt = datetime.strptime(raw_time, "%a %b %d %H:%M:%S %Y")
            post_time = dt.strftime("%Y-%m-%d")
        except:
            post_time = datetime.now().strftime("%Y-%m-%d")

        # 互動計算
        total_reac = like - unlike

        return {
            "platform": "PTT",
            "post_time": post_time,
            "author": author,
            "title": title,
            "content": content,
            "total_reac": total_reac,
            "comment_count": len(push_list),
            "comments_data": push_list
        }

    def parse_article(self, url, author, title):
        res = self.get_response(url)
        if not res:
            return None

        soup = BeautifulSoup(res.text, "lxml")
        main = soup.find("div", id="main-content")

        if not main:
            return None

        # 時間
        try:
            meta = soup.find_all("span", class_="article-meta-value")
            raw_time = meta[3].text.strip()
            # dt = datetime.strptime(raw_time, "%a %b %d %H:%M:%S %Y")
            # article_date = dt.strftime("%Y-%m-%d")
        except:
            raw_time = ""

        
        # 推不推
        like, unlike, arrow = 0, 0, 0
        push_list = []
        pushes = main.find_all("div", class_="push")
        
        # 先把推文取出並轉成標準結構
        for p in pushes:
            try:
                tag = p.find("span", class_="push-tag").text.strip()
                user = p.find("span", class_="push-userid").text.strip()
                comment = p.find("span", class_="push-content").text.strip().lstrip(":")

                if tag == "推":
                    like += 1
                elif tag == "噓":
                    unlike += 1
                else:
                    arrow += 1

                push_list.append({
                    "comment_tag": tag,
                    "comment_author": user,
                    "comment": comment
                })

            except:
                continue
            p.extract()

        # 拆出 meta 資訊與看板標籤
        for meta_tag in main.find_all("div", class_="article-metaline"):
            meta_tag.extract()
        for meta_tag in main.find_all("div", class_="article-metaline-right"):
            meta_tag.extract()
        content = main.text.strip()

        return self.format_data(raw_time, author, title, content, like, unlike, push_list)

    def parse_board(self, board, index):
        url = self.build_board_url(board, index)
        res = self.get_response(url)

        if not res:
            return []

        soup = BeautifulSoup(res.text, "lxml")
        articles = soup.find_all("div", class_="r-ent")

        data = []

        for art in articles:
            title_tag = art.find("div", class_="title").find("a")
            if not title_tag:
                continue

            article_url = self.build_article_url(title_tag["href"])
            title = title_tag.text.strip()

            author = art.find("div", class_="author").text.strip()
            inside = self.parse_article(article_url, author, title)

            if inside:
                # 保留網址以方便未來對照原始網頁
                inside["post_url"] = article_url
                data.append(inside)

            time.sleep(1)
        return data
    
    def run(self, board=None, start_index=1, pages=None):

        if pages is None:
            pages = self.default_pages

        all_data = []

        for i in range(pages):
            index = start_index - i
            print(f"抓第 {i+1} 頁: index{index}")

            page_data = self.parse_board(board, index)
            all_data.extend(page_data)
            time.sleep(2)

        self.save_data(all_data, "result")
        print(f"總共抓取 {len(all_data)} 篇")

        return all_data

# DataCleaner

In [ ]:
import unicodedata

class DataCleaner:
    def __init__(self):
        pass

    def clean(self, platform, data):
        platform_map = {
            "ptt": self.clean_ptt,
            "baha": self.clean_baha,
            "dcard": self.clean_dcard
        }
        clean_func = platform_map.get(platform.lower())
        if not clean_func:
            raise ValueError(f"未知平台，無法清洗: {platform}")
        
        return clean_func(data)

    # Dcard
    def clean_dcard(self, data):
       return self.core_filter_pipeline(data)
    
    # 巴哈
    def clean_baha(self, data):
        return self.core_filter_pipeline(data)
    
    # ptt:擷取文章tag
    def extract_article_tag(self, title):
        if not title:
            return ""

        if "[" in title and "]" in title:
            start = title.find("[")
            end = title.find("]")
            return title[start:end+1]

        return ""

    # PTT push 合併
    def combine_push(self, comments_data):
        if not comments_data:
            return []
        
        combined_comments = []

        temp = {
            "comment_tag": comments_data[0].get("comment_tag", ""),
            "comment_author": comments_data[0].get("comment_author", ""),
            "comment": comments_data[0].get("comment", "")
        }

        for i in range(1, len(comments_data)):
            current = comments_data[i]
            current_author = current.get("comment_author", "")
            current_tag = current.get("comment_tag", "")
            current_comment = current.get("comment", "")

            # 條件：同作者，且當前這條是「箭頭」-> 代表是接續上一條的發言
            if current_author == temp["comment_author"] and current_tag == "→":
                # 直接原地串接文字，不要重設 temp！
                temp["comment"] += current_comment
            else:
                combined_comments.append(temp)
                
                temp = {
                    "comment_author": current_author,
                    "comment": current_comment,
                    "comment_tag": current_tag
                }

        combined_comments.append(temp)

        for item in combined_comments:
            item.pop("comment_tag", None)

        return combined_comments

    # ptt
    def clean_ptt(self, data):
        clean_data = []
        for article in data:
            processed_article = article.copy()
            # # 萃取 tag
            # title = article.get("title", "")
            # article["post_tag"] = self.extract_article_tag(title)

            # 合併 push
            if "comments_data" in processed_article:
                # 先合併推文
                combined = self.combine_push(processed_article.get("comments_data", []))
                
                # 對合併後的每一條 PTT 留言進行文字清洗（包含去網址）
                processed_article["comments_data"] = self.process_and_filter_comments(combined)

            # 修正 overall
            true_overall = processed_article.get("like", 0) - processed_article.get("unlike", 0)
            total_reac = processed_article.get("total_reac", "")

            if total_reac == "" or total_reac is None:
                processed_article["total_reac"] = 0
            elif total_reac == "爆":
                processed_article["total_reac"] = true_overall
            elif (
                isinstance(total_reac, str) and total_reac.startswith("X")
            ):
                processed_article["total_reac"] = true_overall
            elif (
                isinstance(total_reac, str) and total_reac.lstrip("-").isdigit()
            ):
                processed_article["total_reac"] = true_overall

            if "post_url" in processed_article:
                del processed_article["post_url"]

            clean_data.append(processed_article)

        return self.core_filter_pipeline(clean_data)
    
    def process_and_filter_comments(self, comments_list, comment_key="comment"):
        if not isinstance(comments_list, list):
            return []
            
        cleaned_list = []
        for c in comments_list:
            if isinstance(c, dict):
                raw_text = c.get(comment_key, "")
                cleaned_text = self.clean_text(raw_text)
                    
                if cleaned_text and cleaned_text.strip() != "":
                    if len(cleaned_text) >= 5:
                        cleaned_list.append({
                            **c,
                            "comment": cleaned_text
                        })
        return cleaned_list

    def core_filter_pipeline(self, data):
        results = []
        for post in data:
            if not isinstance(post, dict):
                continue

            raw_date = post.get("post_time")
            cleaned_date = self.clean_date(raw_date)
            
            # 只要貼文日期不是 2025 年，整篇貼文跟留言通通不要
            if not cleaned_date or not cleaned_date.startswith("2025"):
                continue
    
            raw_content = post.get("content", "")
            post["content"] = self.clean_text(raw_content)

            post["post_time"] = cleaned_date
            
            raw_comments = post.get("comments_data", [])
            cleaned_comments = self.process_and_filter_comments(raw_comments)
            
            # 如果清洗後留言串變空了，整篇貼文主體也丟棄
            if not cleaned_comments:
                continue
                
            post["comments_data"] = cleaned_comments
            results.append(post)
            
        return results

    # 共用：清洗文字
    def clean_text(self, text):
        if isinstance(text, (list, dict)):
            return text
        if pd.isna(text):
            return None
        text = str(text)

        # 統一將全形英數、全形標點轉為半形
        text = unicodedata.normalize('NFKC', text)

        # 移除網址
        text = text.replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
        text = re.sub(r'https?://[^\s\u4e00-\u9fa5<>""’‘指標“”‘’@!,]+', '', text)
        text = re.sub(r'www\.[^\s\u4e00-\u9fa5<>""’‘指標“”‘’@!,]+', '', text)

        text = re.sub(r'@[a-zA-Z0-9_\.]+(?:/[^\s<>"]*)?(?:\?[^\s<>"]*)?', '', text)
        # 被空白切斷的網址
        text = re.sub(r'https?\s*:\s*/\s*/\S+', '', text,flags=re.IGNORECASE)

        # 移除禮包
        text = re.sub(r'\b(?=[A-Za-z]*\d)(?=\d*[A-Za-z])[A-Za-z0-9]{8,}\b', '', text)
        # 移除好友代碼
        text = re.sub(r'\b\d{12}\b|\b\d{4}\s\d{4}\s\d{4}\b', '', text)
        # 移除巴哈特有的 hot
        text = re.sub(r'^HOT', '', text, flags=re.IGNORECASE)

        # 移除#字號間的所有數字英文與標記
        text = re.sub(r'#[\w:]+#', '', text)
        # 移除[]間所有內容
        text = re.sub(r'\[.*?\]', '', text)
        # 拔除括號顏文字
        text = re.sub(r'\S*?\([^\s)]+?\)\S*?', '', text)

        # 移除表情符號（BERT-Chinese 無法理解）
        text = emoji.replace_emoji(text, replace="")
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])c(?![a-zA-Z0-9])', '', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])c{2,}(?![a-zA-Z0-9])', '', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])0.0(?![a-zA-Z0-9])', '', text)


        # 轉換網路顏文字為對應中文
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])QAQ(?![a-zA-Z0-9])', '好難過', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])QQ(?![a-zA-Z0-9])', '難過', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])orz(?![a-zA-Z0-9])', '無奈倒地', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])lol(?![a-zA-Z0-9])', '大笑', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])TM(?![a-zA-Z0-9])', '他媽', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])XD+(?![a-zA-Z0-9])', '笑', text)

        text = re.sub(r'(?i)(?<![a-zA-Z0-9])[w]{2,}(?!\.)(?![a-zA-Z0-9])', '哈', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])[z]{2,}(?!\.)(?![a-zA-Z0-9])', '睡', text)
        text = re.sub(r'(?i)(?<![a-zA-Z0-9])[w](?![a-zA-Z0-9])', '哈', text)

        text = re.sub(r'=\s*=+|=\s*-+\s*=', '無言', text)
        text = re.sub(r'@@+', '無奈', text)
        text = re.sub(r'><', '害羞', text)
        text = re.sub(r'=\s*3\s*=', '嘟嘴', text)
        text = re.sub(r'(?i):\s*P\b', '', text)
        text = re.sub(r'(?i):\s*D\b', '大笑', text) 
        text = re.sub(r'(?i)=\s*D\b', '大笑', text)        
        
        # 處理 Emoji (若要給 CKIP 斷詞，直接移除)
        text = emoji.replace_emoji(text, replace="")

        # 清空所有特殊雜訊，只留中英數與基本標點
        text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9\s!?,.:;@_=\-+*/~%()\\[\]{}<>]', '', text)

        # 把拔除顏文字後可能留下的「空括號殘渣」清乾淨
        text = re.sub(r'\(\s*\)|\[\s*\]|\{\s*\}|<\s*>', '', text)

        # 收斂中文重複字詞 (例如：啊啊啊啊啊 -> 啊啊)
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)

        # 多餘空白
        text = re.sub(
            r'\s+', ' ', text
        )

        return text.strip()
    
    def process_and_filter_comments(self, comments_list, comment_key="comment"):
        if not isinstance(comments_list, list):
            return []
            
        cleaned_list = []
        for c in comments_list:
            if isinstance(c, dict):
                raw_text = c.get(comment_key, "")
                cleaned_text = self.clean_text(raw_text)
                    
                if (
                cleaned_text
                and cleaned_text.strip() != ""
                and len(cleaned_text.strip()) > 5
                ):
                    cleaned_list.append({
                        **c,
                        "comment": cleaned_text
                })
        return cleaned_list


    # 共用：日期
    def clean_date(self, date_str):
        if pd.isna(date_str):
            return None

        date_str = str(date_str).strip()
        if "年" in date_str and "月" in date_str:
            date_str = (
                date_str
                .replace("年", "-")
                .replace("月", "-")
                .replace("日", "")
            )

        try:
            return (
                pd.to_datetime(date_str).strftime("%Y-%m-%d")
            )
        except:
            return None

# Transform

In [ ]:
import hashlib

class DataConverter:
    def __init__(self):
        pass

    def merge_and_explode_platforms(self, folder_paths: list = None, data_list: list = None) -> pd.DataFrame:

        all_platform_data = []



        # 優先使用直接傳入的資料列表
        if data_list is not None:
            all_platform_data = data_list
        elif folder_paths is not None:
            for folder_path in folder_paths:
                if not os.path.exists(folder_path):
                    print(f"警告：找不到資料夾路徑 {folder_path}")
                    continue
                for filename in os.listdir(folder_path):
                    if not filename.endswith(".json"):
                        continue
                    file_path = os.path.join(folder_path, filename)
                    try:
                        with open(file_path, "r", encoding="utf-8") as f:
                            file_data = json.load(f)
                            if isinstance(file_data, list):
                                all_platform_data.extend(
                                    file_data
                                )

                    except Exception as e:
                        print(f"讀取失敗: {file_path}")
                        print(e)

        if not all_platform_data:
            print("未讀取到任何資料")
            return pd.DataFrame()

        base_df = pd.DataFrame(
            all_platform_data
        )

        required_fields = [
            "platform", "post_time", "author", "title", "content", "total_reac", "comment_count", "comments_data"
        ]

        for field in required_fields:
            if field not in base_df.columns:
                base_df[field] = None

        # comments_data 統一格式
        base_df["comments_data"] = (
            base_df["comments_data"].apply(lambda x: x if isinstance(x, list) else [])
        )

        exploded_df = (
            base_df.explode("comments_data").reset_index(drop=True)
        )

        # 只保留 dict 型態
        valid_comments = exploded_df[
            exploded_df["comments_data"].apply(lambda x:isinstance(x, dict)
            )
        ]

        if not valid_comments.empty:
            comments_normalized = (
                pd.json_normalize(
                    valid_comments[
                        "comments_data"
                    ]
                )
            )

            comments_normalized.index = (
                valid_comments.index
            )
        else:
            comments_normalized = (
                pd.DataFrame(
                    index=exploded_df.index
                )
            )

        exploded_df = exploded_df.drop(
            columns=["comments_data"]
        )

        final_df = pd.concat(
            [exploded_df, comments_normalized
            ],axis=1
        )
        if "comment_author" not in final_df.columns:
            final_df["comment_author"] = np.nan

        if "comment" not in final_df.columns:
            final_df["comment"] = np.nan
        print(
            f"總列數：{len(final_df)}"
        )

        return final_df

    def generate_md5_id(self, platform, post_time, author, title):
        """組合主要欄位計算 MD5"""
        short_title = str(title)[:10] if title else ""
        unique_str = f"{platform}_{post_time}_{author}_{short_title}"
        return hashlib.md5(unique_str.encode('utf-8')).hexdigest()

    def load_json_folder(self, folder_path: str):
        all_data = []
        if not os.path.exists(folder_path):
            print(f"警告：找不到資料夾路徑 {folder_path}")
            return all_data
            
        for filename in os.listdir(folder_path):
            if not filename.endswith(".json"):
                continue
            file_path = os.path.join(folder_path, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    file_data = json.load(f)
                    if isinstance(file_data, list):
                        all_data.extend(file_data)
            except Exception as e:
                print(f"讀取失敗: {file_path}，原因: {e}")
        return all_data

    # 建立資料表架構 (徹底拔除 comment_tag)
    def _create_tables(self, cursor, table_prefix):
        # 貼文主表
        cursor.execute(f'''
            CREATE TABLE IF NOT EXISTS {table_prefix}_posts (
                post_id CHAR(32) PRIMARY KEY,
                platform TEXT,
                post_time TEXT,
                author TEXT,
                total_reac INTEGER,
                title TEXT,
                content TEXT,
                comment_count INTEGER
            )
        ''')
        # 留言副表 (拔除 comment_tag 欄位)
        cursor.execute(f'''
            CREATE TABLE IF NOT EXISTS {table_prefix}_comments (
                comment_id INTEGER PRIMARY KEY AUTOINCREMENT,
                post_id CHAR(32),
                comment_author TEXT,
                comment TEXT,
                FOREIGN KEY (post_id) REFERENCES {table_prefix}_posts (post_id)
            )
        ''')


    # 統一中央處理器 (四個平台共用此邏輯)
    def _insert_generic_data(self, cursor, data, table_prefix):
        self._create_tables(cursor, table_prefix)
        
        for post in data:
            # 計算貼文唯一 ID (用於資料庫防重複)
            post_id = self.generate_md5_id(
                post.get("platform"), 
                post.get("post_time"), 
                post.get("author"), 
                post.get("title")
            )
            
            # 寫入貼文主表
            cursor.execute(f'''
                INSERT OR IGNORE INTO {table_prefix}_posts (
                    post_id, platform, post_time, author, total_reac, title, content, comment_count
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                post_id,
                post.get("platform"),
                post.get("post_time"),
                post.get("author"),
                post.get("total_reac"),
                post.get("title"),
                post.get("content"),
                post.get("comment_count")
            ))
            
            # 展開並寫入留言副表
            comments_data = post.get("comments_data", [])
            for c in comments_data:
                if not isinstance(c, dict):
                    continue
                cursor.execute(f'''
                    INSERT INTO {table_prefix}_comments (
                        post_id, comment_author, comment
                    ) VALUES (?, ?, ?)
                ''', (
                    post_id,
                    c.get("comment_author"),
                    c.get("comment")
                ))


    # 各平台對應分流 (PTT 現在也整合進來了)
    def _sqlite_ptt(self, cursor, data):
        self._insert_generic_data(cursor, data, "ptt")

    def _sqlite_baha(self, cursor, data):
        self._insert_generic_data(cursor, data, "baha")

    def _sqlite_dcard(self, cursor, data):
        self._insert_generic_data(cursor, data, "dcard")


    # 主進入點流程
    def json_to_sqlite(self, folder_path, db_name, platform):
        data = self.load_json_folder(folder_path)
        if not data:
            print(f"[{platform}] 錯誤: 資料夾內無資料或路徑錯誤")
            return
        
        sqlite_process_map = {
            "ptt": self._sqlite_ptt,
            "fb": self._sqlite_fb,
            "baha": self._sqlite_baha,
            "dcard": self._sqlite_dcard
        }

        process_func = sqlite_process_map.get(platform.lower())
        if not process_func:
            raise ValueError(f"未支援該類型平台: {platform}")
            
        try:
            con = sqlite3.connect(db_name)
            cursor = con.cursor()
            cursor.execute("PRAGMA foreign_keys = ON;")
            
            process_func(cursor, data)
            
            con.commit()
            con.close()
            print(f" 成功將 [{platform}] 轉入 SQLite 資料庫: {db_name} (共 {len(data)} 篇貼文)")
        except Exception as e:
            print(f" SQLite 匯入失敗: {e}")

# 爬蟲

## dcard

In [ ]:
# 參數設定
PLATFORM = "dcard"            
RAW_FOLDER = "dcard_data"
OUTPUT_CSV = "Test_cleaned_dcard.csv"

# STEP 1: 執行爬蟲
# ==========================================
print(f"--- 階段 1: 開始爬取 {PLATFORM} 資料 ---")
crawler = dcardcrawler.create(save_folder=RAW_FOLDER)
crawler.run(
    mode="forum", # 需事先確定好網址結構
    query="tower_of_saviors", 
    pages=11, 
    comment_limit=5
)

print(f"\n--- 完成階段1. ---")

# STEP 2: 初始化工具類別
# ==========================================
converter = DataConverter()
cleaner = DataCleaner()

# STEP 3: 資料流串接 (載入 -> 清洗 -> 匯出)
# ==========================================
print(f"\n--- 階段 2: 讀取原始 JSON 資料 ---")
raw_data = converter.load_json_folder(RAW_FOLDER)
print(f"\n--- 完成階段2. ---")

print(f"\n--- 階段 3: 進入 DataCleaner 進行數據清洗 ---")
cleaned_data = cleaner.clean(platform=PLATFORM, data=raw_data)

print(f"\n--- 完成階段3. ---")

print(f"\n--- 階段 4: 進入 DataConverter 匯出結構化資料 ---")

# 轉成 CSV
df_result = converter.merge_and_explode_platforms(data_list=cleaned_data)
if not df_result.empty:
    output_path = OUTPUT_CSV
    # 使用 utf-8-sig 防止產生亂碼
    df_result.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"成功將展開後的資料存檔為: {output_path}")
else:
    print("整合後的 DataFrame 為空，取消存檔。")

# 或者是轉成 SQLite (依需求挑選或並存)
# converter.json_to_sqlite(cleaned_data, db_name=".db", platform=PLATFORM)
print(f"\n--- 完成階段4. ---")

print("\n 全流程執行完畢！")

In [ ]:
# sql 版本
# 參數設定
PLATFORM = "dcard"            
RAW_FOLDER = "dcard_data"
DB_NAME = "PokemonGo_Data.db"

# STEP 1: 執行爬蟲
# ==========================================
print(f"--- 階段 1: 開始爬取 {PLATFORM} 資料 ---")
crawler = dcardcrawler.create(save_folder=RAW_FOLDER)
crawler.run(
    mode="topic", # 需事先確定好網址結構
    query="PokemonGo", 
    pages=11, 
    comment_limit=5
)

print(f"\n--- 完成階段1. ---")

# STEP 2: 初始化工具類別
# ==========================================
converter = DataConverter()
cleaner = DataCleaner()

# STEP 3: 資料流串接 (載入 -> 清洗 -> 匯入 SQL)
# ==========================================
print(f"\n--- 階段 2: 讀取原始 JSON 資料 ---")
raw_data = converter.load_json_folder(RAW_FOLDER)
print(f"--- 完成階段2. ---")

print(f"\n--- 階段 3: 進入 DataCleaner 進行數據清洗 ---")
cleaned_data = cleaner.clean(platform=PLATFORM, data=raw_data)
print(f"--- 完成階段3. ---")

print(f"\n--- 階段 4: 進入 DataConverter 匯入 SQLite 資料庫 ---")

if cleaned_data:
    try:
        # 建立或連接到統一的資料庫
        con = sqlite3.connect(DB_NAME)
        cursor = con.cursor()
        
        # 開啟 SQLite 外鍵支援
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        # 根據平台對應對應的寫入函式 (傳入 cursor 與 清洗後的資料)
        sqlite_process_map = {
            "ptt": converter._sqlite_ptt,
            "fb": converter._sqlite_fb,
            "baha": converter._sqlite_baha,
            "dcard": converter._sqlite_dcard
        }
        
        process_func = sqlite_process_map.get(PLATFORM.lower())
        if process_func:
            process_func(cursor, cleaned_data)
            con.commit()
            print(f" [成功] 已將清洗後的 {PLATFORM} 資料安全匯入資料庫：{DB_NAME}")
        else:
            print(f" [錯誤] 找不到支援的平台處理器: {PLATFORM}")
            
        con.close()
        
    except Exception as e:
        print(f" [失敗] 寫入 SQLite 時發生非預期錯誤: {e}")
else:
    print(f" [警告] 清洗後的資料為空，取消資料庫寫入。")

print(f"\n--- 完成階段4. ---")
print("\n 全流程執行完畢！資料庫已準備就緒。")

## 巴哈

In [ ]:
# 參數設定
PLATFORM = "baha"            
TOPIC = "tower_of_saviors"
RAW_FOLDER = "baha_data"
OUTPUT_CSV = "tower_of_saviors_cleaned_baha.csv"

# STEP 1: 執行爬蟲
# ==========================================
print(f"--- 階段 1: 開始爬取 {PLATFORM} 資料 ---")
crawler = bahacrawler.create(board=TOPIC, save_folder=RAW_FOLDER)
crawler.run(board_id=23805, start_board_page=1, end_board_page=2, reply_pages=2)
print(f"\n--- 完成階段1. ---")

# STEP 2: 初始化工具類別
# ==========================================
converter = DataConverter()
cleaner = DataCleaner()

# STEP 3: 資料流串接 (載入 -> 清洗 -> 匯出)
# ==========================================
print(f"\n--- 階段 2: 讀取原始 JSON 資料 ---")
raw_data = converter.load_json_folder(RAW_FOLDER)
print(f"\n--- 完成階段2. ---")

print(f"\n--- 階段 3: 進入 DataCleaner 進行數據清洗 ---")
cleaned_data = cleaner.clean(platform=PLATFORM, data=raw_data)
print(f"\n--- 完成階段3. ---")

print(f"\n--- 階段 4: 進入 DataConverter 匯出結構化資料 ---")

# 轉成 CSV
df_result = converter.merge_and_explode_platforms(data_list=cleaned_data)
if not df_result.empty:
    output_path = OUTPUT_CSV
    # 使用 utf-8-sig 防止產生亂碼
    df_result.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"成功將展開後的資料存檔為: {output_path}")
else:
    print("整合後的 DataFrame 為空，取消存檔。")
# 或者是轉成 SQLite (依需求挑選或並存)
# converter.json_to_sqlite(cleaned_data, db_name=".db", platform=PLATFORM)
print(f"\n--- 完成階段4. ---")

print("\n 全流程執行完畢！")

## ptt

In [ ]:
# 參數設定
PLATFORM = "ptt"            
TOPIC = "PokemonGO"
RAW_FOLDER = "ptt_data"
OUTPUT_CSV = "PokemonGO_cleaned_ptt.csv"

# STEP 1: 執行爬蟲 (以 PTT 為例)
# ==========================================
print(f"\n==========================\n")
print(f"--- 階段 1: 開始爬取 {PLATFORM} 資料 ---")
crawler = pttcrawler.create(board=TOPIC, save_folder=RAW_FOLDER)
# ptt index是從最後一頁開始，需事先確認好
crawler.run(board=TOPIC, start_index=961, pages=28) # 測試時先註解，避免重複爬取
print(f"\n--- 完成階段1. ---")
print(f"\n==========================\n")

# STEP 2: 初始化工具類別
# ==========================================
converter = DataConverter()
cleaner = DataCleaner()

# STEP 3: 資料流串接 (載入 -> 清洗 -> 匯出)
# ==========================================
print(f"\n--- 階段 2: 讀取原始 JSON 資料 ---")
raw_data = converter.load_json_folder(RAW_FOLDER)
print(f"\n--- 完成階段2. ---")
# print(f"\n==========================\n")

print(f"\n--- 階段 3: 進入 DataCleaner 進行數據清洗 ---")
cleaned_data = cleaner.clean(platform=PLATFORM, data=raw_data)
print(f"\n--- 完成階段3. ---")
# print(f"\n==========================\n")

print(f"\n--- 階段 4: 進入 DataConverter 匯出結構化資料 ---")
# 轉成 CSV
df_result = converter.merge_and_explode_platforms(data_list=cleaned_data)
if not df_result.empty:
    output_path = OUTPUT_CSV
    # 使用 utf-8-sig 防止產生亂碼
    df_result.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"成功將展開後的資料存檔為: {output_path}")
else:
    print("整合後的 DataFrame 為空，取消存檔。")
# 或者是轉成 SQLite (依需求挑選或並存)
# converter.json_to_sqlite(cleaned_data, db_name=".db", platform=PLATFORM)
print(f"\n--- 完成階段4. ---")
print(f"\n==========================\n")

print("\n 全流程執行完畢！")

In [ ]:
# sql 版本
# 參數設定
PLATFORM = "ptt"            
RAW_FOLDER = "ptt_data"
DB_NAME = "PokemonGo_Data.db"

# STEP 1: 執行爬蟲
# ==========================================
print(f"--- 階段 1: 開始爬取 {PLATFORM} 資料 ---")
crawler = pttcrawler.create(board=TOPIC, save_folder=RAW_FOLDER)
# ptt index是從最後一頁開始，需事先確認好
crawler.run(board=TOPIC, start_index=961, pages=28) # 測試時先註解，避免重複爬取
print(f"\n--- 完成階段1. ---")
print(f"\n==========================\n")

# STEP 2: 初始化工具類別
# ==========================================
converter = DataConverter()
cleaner = DataCleaner()

# STEP 3: 資料流串接 (載入 -> 清洗 -> 匯入 SQL)
# ==========================================
print(f"\n--- 階段 2: 讀取原始 JSON 資料 ---")
raw_data = converter.load_json_folder(RAW_FOLDER)
print(f"--- 完成階段2. ---")

print(f"\n--- 階段 3: 進入 DataCleaner 進行數據清洗 ---")
cleaned_data = cleaner.clean(platform=PLATFORM, data=raw_data)
print(f"--- 完成階段3. ---")

print(f"\n--- 階段 4: 進入 DataConverter 匯入 SQLite 資料庫 ---")

if cleaned_data:
    try:
        # 建立或連接到統一的資料庫
        con = sqlite3.connect(DB_NAME)
        cursor = con.cursor()
        
        # 開啟 SQLite 外鍵支援
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        # 根據平台對應對應的寫入函式 (傳入 cursor 與 清洗後的資料)
        sqlite_process_map = {
            "ptt": converter._sqlite_ptt,
            "fb": converter._sqlite_fb,
            "baha": converter._sqlite_baha,
            "dcard": converter._sqlite_dcard
        }
        
        process_func = sqlite_process_map.get(PLATFORM.lower())
        if process_func:
            process_func(cursor, cleaned_data)
            con.commit()
            print(f" [成功] 已將清洗後的 {PLATFORM} 資料安全匯入資料庫：{DB_NAME}")
        else:
            print(f" [錯誤] 找不到支援的平台處理器: {PLATFORM}")
            
        con.close()
        
    except Exception as e:
        print(f" [失敗] 寫入 SQLite 時發生非預期錯誤: {e}")
else:
    print(f" [警告] 清洗後的資料為空，取消資料庫寫入。")

print(f"\n--- 完成階段4. ---")
print("\n 全流程執行完畢！資料庫已準備就緒。")

# 情緒標籤

In [ ]:
db_path = "PokemonGo_Data.db"
con = sqlite3.connect(db_path)

def fetch_platform(con, table_prefix, platform_name, limit=180):
    """
    撈取前 N 筆「熱門貼文的留言」
    """
    query = f"""
    SELECT 
        p.title AS title,
        c.comment AS comment, 
        p.total_reac AS total_reac
    FROM {table_prefix}_comments c
    LEFT JOIN {table_prefix}_posts p ON c.post_id = p.post_id
    WHERE c.comment IS NOT NULL AND c.comment != ''
    ORDER BY CAST(p.total_reac AS INTEGER) DESC 
    LIMIT {limit}
    """
    df = pd.read_sql_query(query, con)
    df['platform'] = platform_name  # 新增來源欄位
    return df

In [ ]:
# 抽樣確認正面用字
current_dir = os.path.abspath("")
db_path = os.path.join(current_dir, "PokemonGo_Data.db")
con = sqlite3.connect(db_path)

positive_keywords = [
    "超讚", "很讚", "超強", "佛心", "真香","畢業", 
    "賺爛", "感謝N社", "感謝s社","高興", 
    "優質", "感動", "好看", "好玩"
]

like_clauses = [f"comment LIKE '%{kw}%'" for kw in positive_keywords]
or_condition = " OR ".join(like_clauses)

platform_list = ["ptt", "baha", "dcard"]
# positive_results = {}
platform = platform_list[0] # 替換要查詢的平台

query = f"""
    SELECT
        p.title     AS title,
        c.comment   AS comment,
        p.total_reac AS total_reac
    FROM {platform}_comments c
    LEFT JOIN {platform}_posts p ON c.post_id = p.post_id
    WHERE c.comment IS NOT NULL AND c.comment != ''
      AND ({or_condition})
    ORDER BY CAST(p.total_reac AS INTEGER) DESC
    """

try:
    pos_df = pd.read_sql_query(query, con)
    print(f"==================================================")
    print(f" 平台 【{platform}】 中潛在的正面留言總筆數：{len(pos_df)} 筆")
    print(f"==================================================")
    
    # 印出前 n 筆來肉眼評估是否真的偏向正面
    if not pos_df.empty:
        print("\n--- 潛在正面留言抽樣預覽 ---")
        for idx, row in pos_df.head(10).iterrows():
            print(f"[{idx+1}] (熱度:{row['total_reac']}) {row['comment']}")
    else:
        print(" 居然連一筆符合關鍵字的留言都沒有，看來社群只有抱怨！")

except Exception as e:
    print(f"SQL 執行失敗，請檢查欄位或連線：{e}")

finally:
    con.close()
    print("\n[提示] 資料庫連線已安全關閉。下一次執行新查詢時，請確保有重新 connect 喔！")


In [ ]:
# 抽樣確認負面用字
current_dir = os.path.abspath("")
db_path = os.path.join(current_dir, "PokemonGo_Data.db")
con = sqlite3.connect(db_path)

negative_keywords = [
    "爛", "超爛", "討厭", "垃圾", 
    "愛錢", "騙錢", "坑錢", "機率低", "bug"
]

like_clauses = [f"comment LIKE '%{kw}%'" for kw in negative_keywords]
or_condition = " OR ".join(like_clauses)
platform = "baha" # 替換要查詢的平台

platform_list = ["ptt", "baha", "dcard"]
# negative_results = {}
# platform = platform_list[0] # 替換要查詢的平台

query = f"""
    SELECT
        p.title     AS title,
        c.comment   AS comment,
        p.total_reac AS total_reac
    FROM {platform}_comments c
    LEFT JOIN {platform}_posts p ON c.post_id = p.post_id
    WHERE c.comment IS NOT NULL AND c.comment != ''
      AND ({or_condition})
    ORDER BY CAST(p.total_reac AS INTEGER) DESC
    LIMIT 30;
    """

try:
    neg_df = pd.read_sql_query(query, con)
    print(f"==================================================")
    print(f" 平台 【{platform}】 中潛在的負面留言總筆數：{len(neg_df)} 筆")
    print(f"==================================================")
    
    # 印出前 n 筆來人肉眼評估是否真的偏向負面
    if not neg_df.empty:
        print("\n--- 潛在負面留言抽樣預覽 ---")
        for idx, row in neg_df.head(10).iterrows():
            print(f"[{idx+1}] (熱度:{row['total_reac']}) {row['comment']}")
    else:
        print(" 居然連一筆符合關鍵字的留言都沒有，請重新檢查！")

except Exception as e:
    print(f"SQL 執行失敗，請檢查欄位或連線：{e}")

finally:
    con.close()
    print("\n[提示] 資料庫連線已安全關閉。下一次執行新查詢時，請確保有重新 connect 喔！")

In [ ]:
current_dir = os.path.abspath("")
db_path = os.path.join(current_dir, "PokemonGo_Data.db")
con = sqlite3.connect(db_path)

neutral_keywords = [
    "普通", "地點", "還好吧", "兌換" #"請問",
    ""
]

like_clauses = [f"comment LIKE '%{kw}%'" for kw in neutral_keywords]
or_condition = " OR ".join(like_clauses)
platform = "ptt" # 替換要查詢的平台
query = f"""
SELECT 
    p.title AS title, 
    c.comment AS comment, 
    p.total_reac AS total_reac
FROM {platform}_comments c
LEFT JOIN {platform}_posts p ON c.post_id = p.post_id
WHERE c.comment IS NOT NULL AND c.comment != ''
AND (
    {or_condition}
)
ORDER BY CAST(p.total_reac AS INTEGER) DESC;
"""

try:
    neutral_df = pd.read_sql_query(query, con)
    print(f"==================================================")
    print(f" 平台 【{platform}】 中潛在的中性留言總筆數：{len(neutral_df)} 筆")
    print(f"==================================================")
    
    # 印出前 n 筆來人肉眼評估是否真的偏向中性
    if not neutral_df.empty:
        print("\n--- 潛在中性留言抽樣預覽 ---")
        for idx, row in neutral_df.head(10).iterrows():
            print(f"[{idx+1}] (熱度:{row['total_reac']}) {row['comment']}")
    else:
        print(" 居然連一筆符合關鍵字的留言都沒有，請重新檢查！")

except Exception as e:
    print(f"SQL 執行失敗，請檢查欄位或連線：{e}")

finally:
    con.close()
    print("\n[提示] 資料庫連線已安全關閉。下一次執行新查詢時，請確保有重新 connect 喔！")

In [ ]:
current_dir = os.path.abspath("")
db_path = os.path.join(current_dir, "PokemonGo_Data.db")
con = sqlite3.connect(db_path)

platform_list = ["ptt", "baha", "dcard"]

# 關鍵字定義
positive_keywords = [
    "超讚", "很讚", "超強", "佛心", "真香", "畢業",
    "賺爛", "感謝N社", "感謝s社", "高興",
    "優質", "感動", "好看", "好玩"
]
negative_keywords = [
    "爛活動", "超爛", "討厭", "垃圾",
    "愛錢", "騙錢", "坑錢", "機率低", "bug"
]
neutral_keywords = [
    "普通", "地點", "還好吧", "兌換"
]

def build_or_condition(keywords):
    return " OR ".join([f"comment LIKE '%{kw}%'" for kw in keywords])

def fetch_comments(con, platform, or_condition, limit=None):
    """查詢單一平台留言，limit=None 表示取全部"""
    limit_clause = f"LIMIT {limit}" if limit else ""
    query = f"""
    SELECT
        '{platform}'   AS platform,
        p.title        AS title,
        c.comment      AS comment,
        p.total_reac   AS total_reac
    FROM {platform}_comments c
    LEFT JOIN {platform}_posts p ON c.post_id = p.post_id
    WHERE c.comment IS NOT NULL AND c.comment != ''
      AND ({or_condition})
    ORDER BY CAST(p.total_reac AS INTEGER) DESC
    {limit_clause};
    """
    return pd.read_sql_query(query, con)

# 各平台查詢
all_frames = []

for platform in platform_list:
    pos_df = fetch_comments(con, platform, build_or_condition(positive_keywords), limit=None)
    pos_df["sentiment"] = "positive"

    neg_df = fetch_comments(con, platform, build_or_condition(negative_keywords), limit=30)
    neg_df["sentiment"] = "negative"

    neu_df = fetch_comments(con, platform, build_or_condition(neutral_keywords), limit=30)
    neu_df["sentiment"] = "neutral"

    all_frames.extend([pos_df, neg_df, neu_df])

con.close()

# 合併並匯出 Excel
combined_df = pd.concat(all_frames, ignore_index=True)

output_path = os.path.join(current_dir, "comments_combined.xlsx")
combined_df.to_excel(output_path, index=False)

print(f"完成！共 {len(combined_df)} 筆，已存至 {output_path}")

In [ ]:
df = pd.read_excel("comments_combined.xlsx")
df["label"] = ""
df["remark"] = ""

# 覆蓋原檔案
df.to_excel("comments_combined.xlsx", index=False)

print("新增完成")

## 第一輪訓練

In [5]:
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, TrainerCallback
from huggingface_hub import InferenceClient
import torch.nn.functional as F
from datasets import Dataset
from transformers import BertTokenizerFast

c:\Users\USER\Desktop\GO_project\venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
HF_token = "hf_YourTokenHere"


# 使用BERT進行文本分類訓練
df_original = pd.read_excel("comments_combined.xlsx")

# 藍位名稱修正以符合 huggingface
df = df_original[['comment', 'label']].copy()
df.columns = ['text', 'label']

train_df, test_df = train_test_split(
    df, test_size = 0.2, random_state = 42,
    stratify = df['label']
    )

# 轉為 Huggingface Dataset 格式
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
full_dataset = Dataset.from_pandas(df)


tokenizer = BertTokenizerFast.from_pretrained('ckiplab/bert-base-chinese')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_full = full_dataset.map(tokenize_function, batched=True)

model = BertForSequenceClassification.from_pretrained('ckiplab/bert-base-chinese', num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    label_smoothing_factor = 0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

print(" 開始訓練 ")
trainer.train()
trainer.save_model("./poc_model")
print("已儲存 ./poc_model")


print("產生預測矩陣中")
predictions = trainer.predict(tokenized_full)
logits = torch.from_numpy(predictions.predictions)

probs = F.softmax(logits, dim=1).numpy()
predict_df = df_original.copy()
predict_df['matrix'] = [f"{p[0]:.2%}, {p[1]:.2%}, {p[2]:.2%}" for p in probs]
predict_df['test_label'] = probs.argmax(axis=1)

predict_df = predict_df[['comment', 'matrix', 'test_label', 'label']]
predict_df.rename(columns={'label': 'true_label'}, inplace=True)

predict_df.to_excel("poc_model_prediction.xlsx", index=False)
print("已成功產出矩陣分析檔案: poc_model_predict180.xlsx")



In [ ]:
print("\n*** loss 歷史紀錄 ***")
for log in trainer.state.log_history:
    epoch = log.get("epoch")
    train_loss = log.get("loss")
    eval_loss = log.get("eval_loss")
    if train_loss or eval_loss:
        print(f"epoch {epoch}, train_loss: {train_loss}, eval_loss: {eval_loss}")

In [ ]:
# 用最佳模型測試

model_path = './poc_model'
tokenizer = BertTokenizer.from_pretrained('ckiplab/bert-base-chinese')
model = BertForSequenceClassification.from_pretrained(model_path)

model.eval() # 確保模型為評估模式
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device) # 將模型移動到適當的設備

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", padding='max_length', truncation=True, max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()} # 將輸入移動到同一設備
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=1)
    predicted_label = torch.argmax(logits, dim=1).item()

    print(f"Logits: {logits.cpu().numpy()}")
    print(f"機率分配: 負:{probs[0][0]:.2%}, 中:{probs[0][1]:.2%}, 正:{probs[0][2]:.2%}")
    
    return predicted_label


print(f" 測試: {predict('這補償好有誠意 超讚')}(期待: 2)")
print(f" 測試: {predict('色違超好看 我一定抓')}(期待: 2)")
print(f" 測試: {predict('新年快樂')}(期待: 1)")
print(f" 測試: {predict('天佑台灣 加油')}(期待: 1)")
print(f" 測試: {predict('這遊戲好爛...')}(期待: 0)")

In [ ]:
# 第一次輸出預測檔案

model_path = os.path.abspath('./poc_model')
input_file = "PokemonGO_cleaned_all.csv"
output_file = "predict_first_round.xlsx"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("loading model...")
tokenizer = BertTokenizerFast.from_pretrained('ckiplab/bert-base-chinese')
model = BertForSequenceClassification.from_pretrained(model_path).to(device)

df = pd.read_csv(input_file)
texts = df['comment'].fillna("").tolist() #確保欄位正確

# 轉格式
test_dataset =Dataset.from_pandas(pd.DataFrame({'text': texts}))

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_dataset = test_dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(output_dir="./temp", per_device_eval_batch_size=32)
trainer = Trainer(model=model, args=training_args)

print(f"start label all comments")
raw_prediction = trainer.predict(tokenized_dataset)
logits = torch.from_numpy(raw_prediction.predictions)

probs = F.softmax(logits, dim=1).numpy()
predicted_labels = probs.argmax(axis=1)

df['predict_label'] = predicted_labels
df['confidence_matrix'] = [f"負:{p[0]:.2%}, 中:{p[1]:.2%}, 正:{p[2]:.2%}" for p in probs]

df.to_excel(output_file, index=False)
print(f"結果儲存至 {output_file}")


In [ ]:
df = pd.read_excel("predict_first_round.xlsx")

# 情感占比
label_count = df['predict_label'].value_counts()
print(label_count)


## 第二輪訓練

In [ ]:
df_original = pd.read_excel("comments_combined.xlsx")
print(df_original['is_generated'].dtype)
print(df_original['is_generated'].unique())

In [ ]:
df_original = pd.read_excel("comments_combined.xlsx")
df = df_original[['comment', 'label', 'is_generated']].copy()
df.columns = ['text', 'label', 'is_generated']

real_df = df[df['is_generated'] == 'F'].copy()
generated_df = df[df['is_generated'] == 'T'].copy()

train_real_df, test_df = train_test_split(
    real_df, test_size=0.2, random_state=42,
    stratify=real_df['label']
)

train_df = pd.concat([train_real_df, generated_df]).sample(frac=1, random_state=42)

train_df = train_df[['text', 'label']].reset_index(drop=True)
test_df = test_df[['text', 'label']].reset_index(drop=True)

In [ ]:
print(f"訓練集：{len(train_df)} 筆")
print(f"測試集：{len(test_df)} 筆")
print(f"訓練集label分布：\n{train_df['label'].value_counts()}")
print(f"測試集label分布：\n{test_df['label'].value_counts()}")

In [ ]:
HF_token = "hf_YourTokenHere"

# 轉為 Huggingface Dataset 格式
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
full_dataset = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

tokenizer = BertTokenizerFast.from_pretrained('ckiplab/bert-base-chinese')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_full = full_dataset.map(tokenize_function, batched=True)

model = BertForSequenceClassification.from_pretrained('ckiplab/bert-base-chinese', num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    label_smoothing_factor = 0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

print(" 開始訓練 ")
trainer.train()
trainer.save_model("./poc_model")
print("已儲存 ./poc_model")


print("產生預測矩陣中")
predictions = trainer.predict(tokenized_full)
logits = torch.from_numpy(predictions.predictions)

probs = F.softmax(logits, dim=1).numpy()
predict_df = df_original.copy()
predict_df['matrix'] = [f"{p[0]:.2%}, {p[1]:.2%}, {p[2]:.2%}" for p in probs]
predict_df['test_label'] = probs.argmax(axis=1)

predict_df = predict_df[['comment', 'matrix', 'test_label', 'label']]
predict_df.rename(columns={'label': 'true_label'}, inplace=True)

predict_df.to_excel("poc_model_prediction240.xlsx", index=False)
print("已成功產出矩陣分析檔案: poc_model_predict240.xlsx")

# 評估模型
# metrics = trainer.evaluate()
# print(f"最終評估指標: {metrics}")



In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

pred = trainer.predict(tokenized_test)

y_true = test_df["label"]

y_pred = pred.predictions.argmax(axis=1)

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="macro"))
print("Recall   :", recall_score(y_true, y_pred, average="macro"))
print("F1       :", f1_score(y_true, y_pred, average="macro"))

print(confusion_matrix(y_true, y_pred))

In [ ]:
df = pd.read_excel("poc_model_prediction240.xlsx")
mask = df["test_label"] != df["true_label"]
wrong_pred = df[mask]
print(f"總共撈出 {len(wrong_pred)} 筆預測錯誤的資料：")
print(wrong_pred)

In [ ]:
df = pd.read_excel("predict_first_round.xlsx")
good_mask = df["predict_label"] == 2
good_pred = df[good_mask]

good_pred.to_csv("positive_file.csv", index=False, encoding="utf-8-sig")

## 第三輪訓練

In [ ]:
df_original = pd.read_excel("comments_combined.xlsx")
df = df_original[['comment', 'label', 'is_generated']].copy()
df.columns = ['text', 'label', 'is_generated']

real_df = df[df['is_generated'] == 'F'].copy()
generated_df = df[df['is_generated'] == 'T'].copy()

train_real_df, test_df = train_test_split(
    real_df, test_size=0.2, random_state=42,
    stratify=real_df['label']
)

train_df = pd.concat([train_real_df, generated_df]).sample(frac=1, random_state=42)

train_df = train_df[['text', 'label']].reset_index(drop=True)
test_df = test_df[['text', 'label']].reset_index(drop=True)

In [ ]:
print(f"訓練集：{len(train_df)} 筆")
print(f"測試集：{len(test_df)} 筆")
print(f"訓練集label分布：\n{train_df['label'].value_counts()}")
print(f"測試集label分布：\n{test_df['label'].value_counts()}")

In [ ]:
HF_token = "hf_YourTokenHere"

# 轉為 Huggingface Dataset 格式
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
full_dataset = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True))

tokenizer = BertTokenizerFast.from_pretrained('ckiplab/bert-base-chinese')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_full = full_dataset.map(tokenize_function, batched=True)

model = BertForSequenceClassification.from_pretrained('ckiplab/bert-base-chinese', num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    label_smoothing_factor = 0.1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

print(" 開始訓練 ")
trainer.train()
trainer.save_model("./poc_model")
print("已儲存 ./poc_model")


print("產生預測矩陣中")
predictions = trainer.predict(tokenized_full)
logits = torch.from_numpy(predictions.predictions)

probs = F.softmax(logits, dim=1).numpy()
predict_df = df_original.copy()
predict_df['matrix'] = [f"{p[0]:.2%}, {p[1]:.2%}, {p[2]:.2%}" for p in probs]
predict_df['test_label'] = probs.argmax(axis=1)

predict_df = predict_df[['comment', 'matrix', 'test_label', 'label']]
predict_df.rename(columns={'label': 'true_label'}, inplace=True)

predict_df.to_excel("poc_model_prediction585.xlsx", index=False)
print("已成功產出矩陣分析檔案: poc_model_predict585.xlsx")


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

pred = trainer.predict(tokenized_test)

y_true = test_df["label"]

y_pred = pred.predictions.argmax(axis=1)

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average="macro"))
print("Recall   :", recall_score(y_true, y_pred, average="macro"))
print("F1       :", f1_score(y_true, y_pred, average="macro"))

print(confusion_matrix(y_true, y_pred))

In [ ]:
df = pd.read_excel("poc_model_prediction585.xlsx")
wrong_neutral = df[
    (df['true_label'] == 1) & 
    (df['test_label'] != 1)
]
print(f"總共撈出 {len(wrong_neutral)} 筆預測錯誤的資料：")
print(wrong_neutral)

## 數據標註

In [ ]:
# 輸出最終預測檔案

model_path = os.path.abspath('./poc_model')
input_file = "PokemonGO_cleaned_all.csv"
output_file = "predict_final.xlsx"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("loading model...")
tokenizer = BertTokenizerFast.from_pretrained('ckiplab/bert-base-chinese')
model = BertForSequenceClassification.from_pretrained(model_path).to(device)

df = pd.read_csv(input_file)
texts = df['comment'].fillna("").tolist() #確保欄位正確

test_dataset =Dataset.from_pandas(pd.DataFrame({'text': texts}))

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_dataset = test_dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(output_dir="./temp", per_device_eval_batch_size=32)
trainer = Trainer(model=model, args=training_args)

print(f"start label all comments")
raw_prediction = trainer.predict(tokenized_dataset)
logits = torch.from_numpy(raw_prediction.predictions)

probs = F.softmax(logits, dim=1).numpy()
predicted_labels = probs.argmax(axis=1)

df['predict_label'] = predicted_labels
df['confidence_matrix'] = [f"負:{p[0]:.2%}, 中:{p[1]:.2%}, 正:{p[2]:.2%}" for p in probs]

df.to_excel(output_file, index=False)
print(f"結果儲存至 {output_file}")

In [ ]:
df = pd.read_excel("predict_final.xlsx")

# 情感占比
label_count = df['predict_label'].value_counts()
print(label_count)

# 文字雲製作

In [ ]:
import jieba
from wordcloud import WordCloud

In [ ]:
df = pd.read_excel("predict_final.xlsx")
df_raw = df.copy()

df_raw["doc_id"] = df_raw.index + 1

pogo_words = ["異色", "色違", "星塵", "復刻", "機率", "bug", "極巨", "背卡"]
for word in pogo_words:
    jieba.add_word(word)

stop_words = {"今天", "又", "了", "根本", "是", "一堆", "太", "啦", "終於", "一"}

def extract_keywords(text):
    if pd.isna(text):
        return []
    words = jieba.lcut(text)
    # 過濾掉停用字、空白字元以及長度小於 2 的字（保留像 bug 這種英文）
    meaningful_words = [
        w
        for w in words
        if w.strip() and w not in stop_words and (len(w) >= 2 or w.isalpha())
    ]
    return meaningful_words

df_raw["keywords"] = df_raw["comment"].apply(extract_keywords)

df_keyword_table = df_raw.explode("keywords")
df_keyword_table = df_keyword_table.dropna(subset=["keywords"])
df_keyword_table = df_keyword_table.rename(columns={"keywords": "Keyword"})

df_output = df_keyword_table[["doc_id", "predict_label", "Keyword"]]

print("--- 攤平後的關鍵字資料表 (準備匯入 Power BI) ---")
print(df_output.head())
df_output.to_csv("pogo_keywords_for_bi.csv", index=False, encoding="utf-8-sig")


power bi 的 Word Cloud 必須是企業版本才能使用，因此文字雲的部分由python完成

In [ ]:
mask = (
    (df['predict_label'] == 0)
)
bad_comment = df.loc[mask, 'comment'].dropna()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 作圖函式

def start_plot(figsize=(10, 8), style = 'whitegrid'):
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(1,1)
    # plt.tight_layout()
    with sns.axes_style(style):
        ax = fig.add_subplot(gs[0,0])
    return ax

def grid_plot(figsize=(18, 14), style = 'whitegrid'):
    fig = plt.figure(figsize=figsize)
    # plt.tight_layout()
    gs=fig.add_gridspec(2, 2)
    with sns.axes_style(style):
        ax0 = fig.add_subplot(gs[0,0])
        ax1 = fig.add_subplot(gs[0,1])
        ax2 = fig.add_subplot(gs[1,0])
        ax3 = fig.add_subplot(gs[1,1])
    return [ax0, ax1, ax2, ax3]

In [ ]:
text = " ".join(bad_comment.astype(str))
font_path = "msjh.ttc" #正黑體

wordcloud = WordCloud(
    font_path=font_path,
    width=800,
    height=400,
    background_color='white'
).generate(text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis('off')
plt.title("Negative Sentiment WordCloud")
plt.show()

In [ ]:
df = pd.read_csv("pogo_keywords_for_bi.csv")

mask = (
    (df['predict_label'] == 0)
)
bad_comment = df.loc[mask, 'Keyword'].dropna()

text = " ".join(bad_comment.astype(str))
font_path = "msjh.ttc" #正黑體

wordcloud = WordCloud(
    font_path=font_path,
    width=800,
    height=400,
    background_color='white'
).generate(text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis('off')
plt.title("Negative Sentiment WordCloud")
plt.show()

In [ ]:
# 停用詞過濾
stopwords = {}.fromkeys(["真的", "不是", "沒有", "可以", "現在", "看到", "什麼", "不會", "只有", "怎麼",
                         "一個", "可能", "這樣", "直接", "還是", "所以", "應該", "就是", "知道", "那個",
                         "這個", "因為", "一樣", "有人", "好像", "如果"])
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white",
    max_words=100,
    contour_width=3,
    contour_color='steelblue',
    font_path=font_path,
    stopwords=stopwords)
wordcloud.generate(' '.join([e for e in jieba.lcut(text) if len(e) >= 2]))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis('off')
plt.title("Negative Sentiment WordCloud")
plt.show()

In [ ]:
mask = (
    (df['predict_label'] == 2)
)
good_comment = df.loc[mask, 'Keyword'].dropna()

text = " ".join(good_comment.astype(str))
font_path = "msjh.ttc" #正黑體

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white",
    max_words=100,
    contour_width=3,
    contour_color='steelblue',
    font_path=font_path,
    stopwords=stopwords)
wordcloud.generate(' '.join([e for e in jieba.lcut(text) if len(e) >= 2]))


plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis('off')
plt.title("Positive Sentiment WordCloud")
plt.show()